# D79: FULL 90 câu bằng **ReAct + Calculator tool** (Qwen3-4B), chuyên đề Mũ-Logarit

Dạng D79: *đếm số nguyên $x$ sao cho ứng với mỗi $x$ có không quá $N$ số nguyên $y$ thỏa mãn
$\log_a(x^2+y)\ge\log_b(x+y)$ (với $a>b>1$)*. Qwen3-4B zero-shot đúng **0/90** ở dạng này (cả 90 câu đều
hết 9.216 token giữa chừng suy luận).

Model **không tự tính tay bất kỳ phép nào**: mọi phép tính đều gọi tool `Calculator` (sympy, chính xác tuyệt đối,
**có bộ nhớ biến**, hỗ trợ `log`, `exp`, `floor`, `ceiling`, `N(...)`, `solve`, `Eq`, so sánh `<`, `>`...) theo vòng
lặp ReAct thật: `Thought → Action → Action Input → Observation → ...`

Prompt gồm 5 phần giống bộ Số phức: `PART_TOOL` (dùng chung) + `PART_KIENTHUC` (kiến thức nền Mũ-Logarit, dùng chung
cho cả chuyên đề) + `PART_HUONGGIAI` (hướng giải riêng của dạng) + `PART_FEWSHOT` (vết ReAct mẫu, các `Observation`
là kết quả chạy thật bằng chính backend ở cell cuối) + `PART_NHIEMVU` (danh sách bước riêng của dạng).

Prompt được **ép mở đầu bằng `<think>\nThought:`** (forced prefix), tham số sinh giữ nguyên bản Số phức
(`temperature=0.3, top_p=0.95, top_k=20, seed=42, presence_penalty=1.2, max_new_tokens=9216`).

**Khác bản Số phức:** `MAX_MODEL_LEN` được tính theo độ dài prompt (prompt dài hơn vì `PART_KIENTHUC` của Mũ-Logarit
lớn hơn; bản Số phức đặt cố định 14336).

## Trước khi chạy

1. Dùng lại Kaggle Dataset chứa `plan_solve_prompts_mulogarit_merged.json` (đã dùng cho zero-shot) và gắn vào notebook
   ("Add Data"). Notebook tự dò đường dẫn bằng `glob`, không cần gõ cứng.
2. Bật GPU (khuyến nghị 2×T4).
3. Để thử vài câu trước khi chạy cả 90 câu, đặt `RUN_STT = [1, 2, 3]` ở ô cấu hình.

Kết quả: `/kaggle/working/d79_react_calculator_full90.csv`
(prompt: 20,048 ký tự, ước lượng khoảng 6,075 token)


In [ ]:
!pip install -q -U vllm
!pip uninstall -y -q torchcodec torchaudio
import vllm; print('vLLM:', vllm.__version__)


In [ ]:
import os, re, json, time, glob
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

MA_DANG = 'D79'
DATA_FILENAME = 'plan_solve_prompts_mulogarit_merged.json'
# The dataset folder name under /kaggle/input differs between accounts, so
# search for the file instead of hardcoding a path.
_matches = glob.glob(f'/kaggle/input/**/{DATA_FILENAME}', recursive=True)
assert _matches, f'{DATA_FILENAME} not found under /kaggle/input. Add the dataset to the notebook first.'
DATA_PATH = _matches[0]
OUT_PATH = f'/kaggle/working/{MA_DANG.lower()}_react_calculator_full90.csv'

RUN_STT = None            # None = all questions; for a quick test use e.g. [1, 2, 3]

MODEL = 'Qwen/Qwen3-4B'
# Same decoding settings as the Complex Numbers ReAct + Calculator runs.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.3, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_NEW_TOKENS = 9216

FREE_UNKNOWNS = ['x']
FREE_UNKNOWNS = set(FREE_UNKNOWNS)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(DATA_PATH, encoding='utf-8') as f:
    records = json.load(f)[MA_DANG]
if RUN_STT is not None:
    records = [r for r in records if r['STT'] in RUN_STT]
print('Questions:', len(records))

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)


In [ ]:
# =====================================================================
# PROMPT: five parts. PART_TOOL and PART_KIENTHUC are shared by every
# problem type of the chapter; PART_HUONGGIAI, PART_FEWSHOT and
# PART_NHIEMVU are specific to this one.
# =====================================================================

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: rate = log(6)/log(5)` → `Observation: rate = log(6)/log(5)`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: big = N(101**rate - 101, 20)`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Power is `**` (e.g. `x**2`, `101**rate`), not `^`. Square root is `sqrt(...)`; never the `√` symbol.
- Plain fractions are already exact: writing `7/2` gives exactly seven halves, never a decimal. You do NOT need any special function for fractions.
- Logarithms: `log(u)` is the NATURAL logarithm (write `log`, never `ln`). A logarithm to another base is written with the change of base formula, `log(u)/log(base)`, or equivalently `log(u, base)`. The exponential function is `exp(u)` and Euler's number is `E`.
- Absolute value is `Abs(u)`. Rounding: `floor(u)` is the greatest integer not above `u`, and `ceiling(u)` is the least integer not below `u`. `Min(u, v)` and `Max(u, v)` give the smaller and the larger of two numbers.
- Exact results stay exact (for example `log(8)/log(2)` becomes `3`). When you need a decimal value, for instance to see between which integers a number lies, write `N(expression, 20)`: this is the Calculator computing a decimal for you, not you approximating.
- To solve an equation, write `solve(expression, unknown)`, where the expression is the left side of the equation once everything is moved to one side so that it equals 0. It returns the list of solutions. The letter `x` may be used as an unknown inside `solve`, `expand`, `factor` and `subs` without being assigned first. Other tools: `expand(...)`, `factor(...)`, `diff(f, x)`, and `expr.subs(x, value)` to substitute.
- To test whether two expressions are **exactly** equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`, decided exactly; never approximately. To compare two numbers by size, write `left < right`, `left <= right`, `left > right` or `left >= right`; the Calculator decides it with 50 digits of precision and answers `True` or `False`. Never write `==`.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation solving mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non real ones; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`) and discard it immediately if not. For a root confirmed real, use `re(root)`, not the raw value. Any comparison built on numeric roots should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root, in particular the final count and matching it against the answer options, stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''

PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF EXPONENTIALS AND LOGARITHMS**

1. **Powers**: for $a>0$ and real $m,n$: $a^m a^n=a^{m+n}$, $\dfrac{a^m}{a^n}=a^{m-n}$, $(a^m)^n=a^{mn}$, $(ab)^n=a^n b^n$, $a^{-n}=\dfrac{1}{a^n}$, $a^0=1$, and $a^{m/n}=\sqrt[n]{a^m}$.
2. **Exponential function** $y=a^x$ with $a>0$, $a\ne1$: it is defined for every real $x$, and $a^x>0$ always (it never equals 0 and is never negative). It is strictly increasing when $a>1$ and strictly decreasing when $0<a<1$.
3. **Exponential equations**: $a^{f}=a^{g}\iff f=g$. For a number $c$: $a^{f}=c$ has a solution only when $c>0$, and then $f=\log_a c$.
4. **Exponential inequalities** (same base): if $a>1$, then $a^{f}<a^{g}\iff f<g$ (direction kept). If $0<a<1$, then $a^{f}<a^{g}\iff f>g$ (direction reversed). In particular the sign of $a^{u}-a^{v}$ equals the sign of $u-v$ when $a>1$.
5. **Substitution $t=a^x$**: because $a^x>0$, the new variable must satisfy $t>0$. Then $a^{2x}=t^2$ and $a^{x+k}=a^k\,t$.
6. **Definition of the logarithm**: for $a>0$, $a\ne1$, $b>0$: $\log_a b=c\iff a^c=b$. The base must be positive and not equal to 1, and the argument must be strictly positive.
7. **Basic values**: $\log_a 1=0$, $\log_a a=1$, $\log_a a^c=c$, and $a^{\log_a b}=b$ for $b>0$.
8. **Product, quotient, power** (for $x,y>0$): $\log_a(xy)=\log_a x+\log_a y$, $\log_a\dfrac{x}{y}=\log_a x-\log_a y$, $\log_a x^{n}=n\log_a x$. For an even power of a number $x\ne0$ that may be negative: $\log_a x^{2k}=2k\log_a|x|$. Also $\log_{a^m}x=\dfrac{1}{m}\log_a x$ and $\log_a\sqrt[n]{x}=\dfrac{1}{n}\log_a x$.
9. **Change of base**: $\log_a b=\dfrac{\log_c b}{\log_c a}$ for any base $c>0$, $c\ne1$. Consequences: $\log_a b=\dfrac{1}{\log_b a}$ (for $b\ne1$) and $\log_a b\cdot\log_b c=\log_a c$.
10. **Natural and common logarithm**: $\ln x=\log_e x$ and $\log x=\log_{10}x$. In particular $\log_a b=\dfrac{\ln b}{\ln a}$.
11. **Exchanging base and exponent**: $a^{\log_b c}=c^{\log_b a}$ for $a,c>0$ and $b>0$, $b\ne1$. Reason: taking $\log_b$ of each side gives $\log_b c\cdot\log_b a$ on both sides, and two positive numbers with the same logarithm are equal.
12. **Logarithmic function** $y=\log_a x$: its domain is $x>0$. It is strictly increasing when $a>1$ and strictly decreasing when $0<a<1$. If $a>1$: $\log_a u<\log_a v\iff 0<u<v$ (direction kept). If $0<a<1$: $\log_a u<\log_a v\iff u>v>0$ (direction reversed). In both cases the arguments must be positive first.
13. **Logarithmic equations**: first state the domain (every argument $>0$). Then $\log_a f=\log_a g\iff f=g$, and $\log_a f=c\iff f=a^c$. A solution is accepted only if it satisfies the domain.
14. **Logarithmic inequalities with a constant**: if $a>1$: $\log_a f>c\iff f>a^c$ and $\log_a f<c\iff 0<f<a^c$. If $0<a<1$: $\log_a f>c\iff 0<f<a^c$ and $\log_a f<c\iff f>a^c$.
15. **Substitution $t=\log_a x$**: here $x>0$ and $t$ can be any real number. Then $\log_a^2 x=t^2$ and $x=a^t$.
16. **Always find the domain first**: every logarithm argument $>0$, every base positive and not 1, every denominator $\ne0$, every expression under a square root $\ge0$. Any value found later is accepted only if it satisfies this domain.
17. **Sign of a difference of two logarithms** (base $a>1$, arguments $u,v>0$): the sign of $\log_a u-\log_a v$ equals the sign of $u-v$. In particular $\log_a u-\log_a v=0\iff u=v$.
18. **Products and square roots**: $A\cdot B=0$ means $A=0$ or $B=0$. For $A\cdot\sqrt{B}=0$: either $A=0$ (and then $B\ge0$ must hold so that the square root exists), or $B=0$. The sign of a product $A\cdot B$ is decided by the signs of its two factors: $A\cdot B\le 0$ means one factor is $\ge0$ and the other is $\le0$.
19. **Derivatives**: $(a^x)'=a^x\ln a$, $(e^x)'=e^x$, $(a^u)'=u'\,a^u\ln a$. Also $(\log_a x)'=\dfrac{1}{x\ln a}$, $(\ln x)'=\dfrac{1}{x}$, $(\log_a u)'=\dfrac{u'}{u\ln a}$, and for a quotient $\left(\dfrac{u}{v}\right)'=\dfrac{u'v-uv'}{v^2}$.
20. **Derivative and monotonicity**: if $f'(x)>0$ on an interval, $f$ is strictly increasing there; if $f'(x)<0$, it is strictly decreasing. A strictly increasing function satisfies $f(u)<f(v)\iff u<v$ and $f(u)=f(v)\iff u=v$, so the equation $f=k$ has at most one solution.
21. **A power function with exponent above 1**: if $\alpha>1$ then $f(t)=t^{\alpha}-t$ has $f'(t)=\alpha t^{\alpha-1}-1$, and for $t\ge1$ we get $t^{\alpha-1}\ge1$, so $f'(t)\ge\alpha-1>0$: $f$ is strictly increasing on $[1,+\infty)$.
22. **Comparing the bases**: if $a>b>1$ then $\log_b a=\dfrac{\ln a}{\ln b}>1$.
23. **Integers**: a sum or difference of integers is an integer, and a positive integer is at least 1. Different integer values of $t$ give different values of $y=t-x$ for a fixed integer $x$, so counting the integers $y$ is the same as counting the integers $t$.
24. **Counting integers**: for real numbers $p<q$, the number of integers strictly between them (open interval $(p,q)$) is $(\lceil q\rceil-1)-(\lfloor p\rfloor+1)+1=\lceil q\rceil-\lfloor p\rfloor-1$. The number of integers in the closed interval $[p,q]$ is $\lfloor q\rfloor-\lceil p\rceil+1$. Here $\lfloor u\rfloor$ is the greatest integer not above $u$ and $\lceil u\rceil$ is the least integer not below $u$. The integers $1,2,\dots,T$ are exactly $T$ numbers.
25. **Quadratic equation** $ax^2+bx+c=0$ ($a\ne0$): $\Delta=b^2-4ac$. If $\Delta>0$ there are two distinct real roots $x_{1,2}=\dfrac{-b\pm\sqrt{\Delta}}{2a}$; if $\Delta=0$ one double root; if $\Delta<0$ no real root. Viète: $x_1+x_2=-\dfrac{b}{a}$ and $x_1x_2=\dfrac{c}{a}$.
26. **Quadratic inequality**: for $a>0$ and $\Delta>0$ with roots $x_1<x_2$: $ax^2+bx+c<0\iff x_1<x<x_2$, and $ax^2+bx+c>0\iff x<x_1$ or $x>x_2$. The two roots themselves are included only for the non strict signs $\le$ and $\ge$.
27. **Two distinct positive roots**: the quadratic $at^2+bt+c=0$ has two distinct positive roots if and only if $\Delta>0$, $x_1+x_2>0$ and $x_1x_2>0$.
28. **Squares and square roots**: $\sqrt{u^2}=|u|$. For $k>0$: $x^2<k\iff-\sqrt{k}<x<\sqrt{k}$ and $x^2>k\iff x<-\sqrt{k}$ or $x>\sqrt{k}$.
29. **Inequality of the arithmetic and geometric means**: for $u,v\ge0$: $u+v\ge2\sqrt{uv}$, with equality if and only if $u=v$.
30. **Bunyakovsky inequality** (also called Cauchy and Schwarz): for real numbers $p,q,x,y$: $(px+qy)^2\le(p^2+q^2)(x^2+y^2)$, with equality if and only if $(p,q)$ and $(x,y)$ are proportional.
'''

PART_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: the problem gives two logarithms with different bases $a>b>1$ and a bound $N$, and asks how many integers $x$ have the following property: for this $x$, there are at most $N$ integers $y$ satisfying $\log_a(x^2+y)\ge\log_b(x+y)$.

**Domain.** Both logarithms need a positive argument, so we need $x+y>0$ and $x^2+y>0$. The numbers $x$ and $y$ are integers, so $x+y$ is an integer, and a positive integer is at least 1. The first condition therefore says $x+y\ge1$.

**A new variable that separates $x$ from $y$.** The two arguments share the sum $x+y$, so we give it a name: $t=x+y$, an integer with $t\ge1$. From $y=t-x$, the other argument becomes $x^2+y=x^2-x+t$. The inequality is now $\log_a(x^2-x+t)\ge\log_b t$.

**Separating the two variables.** The aim is to have only $x$ on one side and only $t$ on the other. Since $a>1$, the function $u\mapsto a^u$ is strictly increasing, so raising $a$ to both sides keeps the direction of the inequality: $x^2-x+t\ge a^{\log_b t}$. The power on the right has $t$ inside the exponent, and exchanging base and exponent turns it into $a^{\log_b t}=t^{\log_b a}$. Moving $t$ to the right side gives
$$x^2-x\ \ge\ t^{\log_b a}-t.$$
Now $x$ appears only on the left and $t$ only on the right.

**Studying the right side.** Let $f(t)=t^{\log_b a}-t$. Because $a>b>1$, the exponent $\log_b a$ is greater than 1, so $f'(t)=\log_b a\cdot t^{\log_b a-1}-1\ge\log_b a-1>0$ for $t\ge1$. Hence $f$ is strictly increasing on $[1,+\infty)$: as $t$ takes the values $1,2,3,\dots$, the value $f(t)$ only grows.

**Turning "at most $N$ values of $y$" into a condition on $x$.** For a fixed integer $x$, each integer $t\ge1$ gives exactly one integer $y=t-x$, so counting the integers $y$ is the same as counting the integers $t$ with $f(t)\le x^2-x$. Because $f$ increases, these values of $t$ form an unbroken run $1,2,\dots,T$: once $f(t)$ rises above $x^2-x$, it stays above. This run has at most $N$ members exactly when the next integer $N+1$ already breaks the inequality, that is when $f(N+1)>x^2-x$. Indeed, if $N+1$ fails, every larger integer fails as well; and if $N+1$ does not fail, the run reaches at least $N+1$ and has too many members. So the property required of $x$ is
$$x^2-x<f(N+1).$$

**Conclusion.** Putting $N+1$ into $f$ gives one definite real number $C=f(N+1)=(N+1)^{\log_b a}-(N+1)$. The condition on $x$ becomes $x^2-x-C<0$. This quadratic has a positive leading coefficient and two real roots $x_1<x_2$ (its discriminant $1+4C$ is positive), so it is negative exactly for $x_1<x<x_2$. Since $x$ is an integer, the answer is the number of integers strictly between $x_1$ and $x_2$. This count is then compared with the four options.
'''

PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Có bao nhiêu số nguyên $x$ sao cho ứng với mỗi $x$ có không quá 120 số nguyên $y$ thỏa mãn $\log_{7}(x^2+y)\ge \log_{5}(x+y)$?

A. $27$.
B. $29$.
C. $28$.
D. $30$.

<think>
Thought: I read the problem. The first logarithm is $\log_7(x^2+y)$, so its base is 7; the second is $\log_5(x+y)$, so its base is 5; and the bound is 120 integers $y$. The method says the decisive integer is the bound plus 1, the first value of $t$ that must fail, so I ask the Calculator for it first.
Action: Calculator
Action Input: n_plus_1 = 120 + 1
Observation: n_plus_1 = 121

Thought: After the substitution $t=x+y$ the inequality became $x^2-x\ge t^{\log_5 7}-t$, so the exponent I need is $\log_5 7$, the base of the first logarithm written as a logarithm to the base of the second. The Calculator only has natural logarithms, so I use the change of base formula $\log(7)/\log(5)$.
Action: Calculator
Action Input: exponent = log(7)/log(5)
Observation: exponent = log(7)/log(5)

Thought: Now the threshold $C=f(121)=121^{\log_5 7}-121$. I need it as a decimal, because the next step must locate two roots between consecutive integers, so I wrap the expression in N with 20 digits.
Action: Calculator
Action Input: threshold = N(n_plus_1**exponent - n_plus_1, 20)
Observation: threshold = 208.77410620563734142

Thought: The condition on $x$ is $x^2-x<C$, so I solve the quadratic equation $x^2-x-C=0$. Its leading coefficient is positive, so the inequality will hold strictly between the two roots.
Action: Calculator
Action Input: x_roots = solve(x**2 - x - threshold, x)
Observation: x_roots = [-13.9576660013170, 14.9576660013170]

Thought: The solver may list the two roots in either order, so I do not rely on their positions. I store the smaller root by taking the minimum of the two.
Action: Calculator
Action Input: x_low = Min(x_roots[0], x_roots[1])
Observation: x_low = -13.9576660013170

Thought: Now the larger root, taking the maximum of the two.
Action: Calculator
Action Input: x_high = Max(x_roots[0], x_roots[1])
Observation: x_high = 14.9576660013170

Thought: The integers $x$ must satisfy $x_{low}<x<x_{high}$ strictly. The first integer strictly above $x_{low}$ is the floor of $x_{low}$ plus 1; this stays correct even if $x_{low}$ were itself an integer, because that value would then be excluded.
Action: Calculator
Action Input: first_x = floor(x_low) + 1
Observation: first_x = -13

Thought: The last integer strictly below $x_{high}$ is the ceiling of $x_{high}$ minus 1, for the same reason.
Action: Calculator
Action Input: last_x = ceiling(x_high) - 1
Observation: last_x = 14

Thought: The integers from first_x to last_x inclusive are last_x minus first_x plus 1 numbers, and this is the number of valid $x$.
Action: Calculator
Action Input: count = last_x - first_x + 1
Observation: count = 28

Thought: The count is stored. Before testing anything, I copy all four options exactly as they are written in the problem, side by side: A: 27, B: 29, C: 28, D: 30. I will test each of them against the count, one call per option, and read each result next to the option I copied here.

Thought: Option A is 27.
Action: Calculator
Action Input: test_A = Eq(count, 27)
Observation: test_A = False

Thought: Option B is 29.
Action: Calculator
Action Input: test_B = Eq(count, 29)
Observation: test_B = False

Thought: Option C is 28.
Action: Calculator
Action Input: test_C = Eq(count, 28)
Observation: test_C = True

Thought: Option D is 30.
Action: Calculator
Action Input: test_D = Eq(count, 30)
Observation: test_D = False

Thought: Lining up what I copied with the stored results: A: 27 gives False, B: 29 gives False, C: 28 gives True, D: 30 gives False. Exactly one option matches, and it is C.

</think>
Final Answer: \boxed{C}
'''

PART_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Read the problem and write down, without computing: the base $a$ of the first logarithm $\log_a(x^2+y)$, the base $b$ of the second logarithm $\log_b(x+y)$, and the bound, which is the number in the phrase "at most ... integers $y$".
[Step 2] Store `n_plus_1 = <the bound> + 1`, writing the number from the problem in place of `<the bound>`.
[Step 3] Store `exponent = log(a)/log(b)`, writing the two bases from the problem. This is $\log_b a$ by the change of base formula.
[Step 4] Store `threshold = N(n_plus_1**exponent - n_plus_1, 20)`. This is the real number $C=f(N+1)$.
[Step 5] Store `x_roots = solve(x**2 - x - threshold, x)`.
[Step 6] Store `x_low = Min(x_roots[0], x_roots[1])` and then `x_high = Max(x_roots[0], x_roots[1])`.
[Step 7] Store `first_x = floor(x_low) + 1`, then `last_x = ceiling(x_high) - 1`, then `count = last_x - first_x + 1`.
[Step 8] Copy all four options side by side in one Thought. Then test every option in order with `test_A = Eq(count, <option A>)`, then `test_B`, `test_C`, `test_D` in the same way. Finally line up each copied option next to its stored result; exactly one of them is `True`. Do not pick the letter by eye.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''

PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)


In [ ]:
# The context window must hold the prompt plus the whole generation budget.
longest = max(records, key=lambda r: len(r['de_bai_mcq']))
PROMPT_TOKENS = len(tok(PROMPT_TEMPLATE.replace('{de_bai}', longest['de_bai_mcq'])).input_ids)
MAX_MODEL_LEN = ((PROMPT_TOKENS + MAX_NEW_TOKENS + 1500) // 1024 + 1) * 1024
print(f'Prompt: {PROMPT_TOKENS} tokens -> MAX_MODEL_LEN={MAX_MODEL_LEN}')

N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'GPUs: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
print('Ready.')


In [ ]:
# =====================================================================
# BACKEND: the Calculator tool (SymPy with variable memory)
# =====================================================================
import re

import sympy as sp

ASSIGN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)
# Names that already mean something to the Calculator (functions such as N, Min, floor and
# constants such as E, I, S). Storing a value under one of them would silently replace it,
# for example "N = 94" would make the later call N(...) fail for the rest of the question.
RESERVED_NAMES = {n for n in dir(sp) if not n.startswith('_')}
RELATIONAL = (sp.StrictLessThan, sp.LessThan, sp.StrictGreaterThan, sp.GreaterThan, sp.Unequality)


class Calculator:
    """SymPy calculator with variable memory (one fresh instance per question)."""

    def __init__(self, free_unknowns=()):
        self.free_unknowns = set(free_unknowns)
        # Free unknowns are registered up front as REAL symbols; letting
        # sympify create them lazily would make them generic complex symbols.
        self.ns = {name: sp.Symbol(name, real=True) for name in self.free_unknowns}

    def _undefined_names(self, expr):
        free = getattr(expr, 'free_symbols', set())
        missing = {str(s) for s in free} - self.free_unknowns
        if not missing:
            return None
        unknown = ', '.join(sorted(missing))
        in_memory = ', '.join(sorted(self.ns)) or '(nothing stored yet)'
        allowed = ', '.join(sorted(self.free_unknowns)) or '(none)'
        return (f'undefined name(s): {unknown}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {in_memory}. Declared free unknowns '
                f'(allowed without prior assignment): {allowed}. Either compute '
                'and store that name first (with "name = expression"), or '
                'rewrite the expression without it.')

    def _unknown_functions(self, parsed):
        """Names used as functions that the Calculator does not know (ceil, log10, int, ...).

        SymPy accepts any unknown name followed by parentheses and silently keeps it as an
        unevaluated symbol, so "count = ceil(115.36) - 1" would be stored as if it were a
        result. Refuse it with a message that says what to write instead."""
        items = parsed if isinstance(parsed, (list, tuple, sp.FiniteSet)) else [parsed]
        names = set()
        for item in items:
            for node in getattr(item, 'atoms', lambda *a: set())(sp.core.function.AppliedUndef):
                names.add(type(node).__name__)
        if not names:
            return None
        return (f'unknown function(s): {", ".join(sorted(names))}. The Calculator does not know '
                'these names, so nothing was computed. Use only: log, exp, sqrt, floor, '
                'ceiling, Abs, Min, Max, N, solve, expand, factor, diff, nroots, Eq. '
                'For example: round up with ceiling(u), round down with floor(u), a logarithm '
                'to base b as log(u)/log(b).')

    def _decide_relational(self, rel):
        """True or False for an inequality between constant expressions."""
        err = self._undefined_names(rel.lhs - rel.rhs)
        if err:
            return None, err
        if rel.lhs.free_symbols or rel.rhs.free_symbols:
            return None, ('cannot decide an inequality that still contains an '
                          'unknown. Substitute numbers first.')
        diff = sp.N(rel.lhs - rel.rhs, 50)
        if abs(diff) < sp.Float('1e-40'):
            diff = sp.Integer(0)
        table = {sp.StrictLessThan: diff < 0, sp.LessThan: diff <= 0,
                 sp.StrictGreaterThan: diff > 0, sp.GreaterThan: diff >= 0,
                 sp.Unequality: diff != 0}
        return ('True' if table[type(rel)] else 'False'), None

    def _boolean_result(self, name, truth):
        """Format a True/False answer; store it when it was assigned to a name."""
        text = 'True' if truth else 'False'
        if name:
            self.ns[name] = sp.true if truth else sp.false
            return f'{name} = {text}', None
        return text, None

    def evaluate(self, expression):
        """Return (result_text, error_text). Exactly one of them is None."""
        s = expression.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        name = None
        m = ASSIGN_RE.match(s)
        # "x = Eq(...)" is an assignment; a bare "Eq(a, b)" has no top level "=".
        if m and not s.lstrip().startswith('Eq('):
            name, s = m.group(1), m.group(2)
            if name in self.free_unknowns:
                return None, (f'"{name}" is a declared unknown used by solve(...) and cannot be '
                              'a variable name. Choose another name and store the value again.')
            if name in RESERVED_NAMES:
                return None, (f'"{name}" cannot be used as a variable name because it is the name '
                              'of a Calculator function or constant (for example N(expression, '
                              'digits) computes a decimal value). Choose another name, such as a '
                              'descriptive word, and store the value again.')

        try:
            parsed = sp.sympify(s, locals=self.ns)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        err = self._unknown_functions(parsed)
        if err:
            return None, err

        try:
            if isinstance(parsed, sp.Equality):
                err = self._undefined_names(parsed.lhs - parsed.rhs)
                if err:
                    return None, err
                same = sp.simplify(parsed.lhs - parsed.rhs) == 0
                return self._boolean_result(name, same)

            if isinstance(parsed, RELATIONAL):
                text, err = self._decide_relational(parsed)
                if err:
                    return None, err
                return self._boolean_result(name, text == 'True')

            if isinstance(parsed, (sp.logic.boolalg.BooleanTrue, sp.logic.boolalg.BooleanFalse)):
                return self._boolean_result(name, bool(parsed))

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Result of solve(...): a list of solutions. Check each element
                # separately instead of simplifying the list as a whole.
                items = list(parsed)
                for item in items:
                    err = self._undefined_names(item)
                    if err:
                        return None, err
                if name:
                    self.ns[name] = items
                    return f'{name} = {items}', None
                return str(items), None

            value = sp.expand(sp.simplify(parsed))
            err = self._undefined_names(value)
            if err:
                return None, err
            if name:
                self.ns[name] = value
                return f'{name} = {value}', None
            return str(value), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'


# =====================================================================
# BATCHED ReAct LOOP: one vLLM call per round for every active question,
# then the Calculator runs separately for each question.
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 25
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')


class QuestionState:
    """ReAct state of ONE question (own variable memory, own token budget)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.calculator = Calculator(free_unknowns=FREE_UNKNOWNS)
        # Forced prefix: the model must continue from "Thought:" right after
        # <think>, it cannot start with free prose.
        self.full_text = '<think>\nThought:'
        self.total_tokens = 0
        self.n_calls = 0
        self.tool_log = []
        self.done = False
        self.stop_reason = ''
        self.final_round = False          # tool limit reached: force the Final Answer


initial_prompts = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    initial_prompts.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

states = [QuestionState(r, p) for r, p in zip(records, initial_prompts)]

t0 = time.time()
round_no = 0
while True:
    active = [s for s in states if not s.done]
    if not active:
        break
    round_no += 1

    batch = []
    for s in active:
        model_input = s.chat_prompt + s.full_text
        room = MAX_MODEL_LEN - len(tok(model_input).input_ids) - 8
        n_new = min(MAX_NEW_TOKENS - s.total_tokens, room)
        if n_new <= 0:
            s.done = True
            s.stop_reason = 'out_of_context' if room <= 0 else 'out_of_token_budget'
            continue
        stop = None if s.final_round else [STOP_STR]
        batch.append((s, model_input, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=n_new,
            seed=SEED, stop=stop)))

    if not batch:
        break

    outs = llm.generate([p for _, p, _ in batch],
                        [sp_ for _, _, sp_ in batch], use_tqdm=False)

    calls_this_round = 0
    for (s, _, _), out in zip(batch, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.total_tokens += len(o.token_ids)

        if s.final_round:
            s.done = True
            s.stop_reason = 'tool_limit_reached'
            continue

        if o.stop_reason != STOP_STR:
            s.done = True
            s.stop_reason = ('model_finished' if o.finish_reason == 'stop'
                             else 'out_of_tokens')
            continue

        s.n_calls += 1
        calls_this_round += 1
        match = None
        for mm in ACTION_INPUT_RE.finditer(o.text):
            match = mm
        expression = match.group(1).strip() if match else ''

        if not expression:
            observation = ('ERROR: no "Action Input:" line found. Write a Thought, '
                           'then "Action: Calculator", then "Action Input: <expression>".')
            s.tool_log.append(('(no Action Input)', observation))
        else:
            result, error = s.calculator.evaluate(expression)
            if error:
                observation = (f'ERROR: {error}. Fix the syntax and try again '
                               '(sqrt() for roots, ** for powers, log() for logarithms, '
                               'Eq(a, b) to compare).')
                s.tool_log.append((expression, f'ERROR: {error}'))
            else:
                observation = result
                s.tool_log.append((expression, result))

        s.full_text += f'{STOP_STR} {observation}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached, give your Final '
                            'Answer now, with no further Action.]\n')
            s.final_round = True

    print(f'  Round {round_no:2d}: {len(batch):2d} questions generating, '
          f'{calls_this_round:2d} tool calls, '
          f'{sum(1 for s in states if not s.done):2d} still active '
          f'({time.time()-t0:.0f}s)')

print(f'Finished after {round_no} rounds, {(time.time()-t0)/60:.1f} minutes.')


# =====================================================================
# GRADING + SAVING
# =====================================================================
def extract_letter(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*(?:\\text\{\s*)?([A-D])\b',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in states:
    chosen = extract_letter(s.full_text)
    n_err = sum(1 for _, obs in s.tool_log if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'ma_dang': s.rec.get('ma_dang', MA_DANG),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chosen,
        'DUNG': chosen == s.rec['dap_an_letter'],
        'token': s.total_tokens,
        'tool_calls': s.n_calls,
        'syntax_errors': n_err,
        'stop_reason': s.stop_reason,
        'tool_log': ' | '.join(f'{ex} -> {obs}' for ex, obs in s.tool_log),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

n_right = int(df['DUNG'].sum())
n_none = int((df['model_chon'] == '').sum())
print('=' * 72)
print(f"RESULT: {n_right} / {len(df)} correct ({df['DUNG'].mean()*100:.1f}%)   "
      f"wrong={len(df) - n_right - n_none}   undetermined={n_none}")
print('=' * 72)
print()
print('--- Tool usage quality ---')
print(f"Average tool calls per question : {df['tool_calls'].mean():.1f}")
print(f"Total tool calls                : {df['tool_calls'].sum()}")
print(f"Total syntax errors             : {df['syntax_errors'].sum()} "
      f"({df['syntax_errors'].sum() / max(1, df['tool_calls'].sum()) * 100:.1f}%)")
print(f"Questions with NO tool call     : {(df['tool_calls'] == 0).sum()}")
print(f"Average tokens per question     : {df['token'].mean():.0f}")
print()
print('--- Stop reason ---')
print(df['stop_reason'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- WRONG OR UNDETERMINED QUESTIONS ---')
    print(df[~df['DUNG']][['STT', 'dap_an_dung', 'model_chon', 'token',
                           'tool_calls', 'syntax_errors', 'stop_reason']].to_string(index=False))
else:
    print(f"*** ALL {len(df)} QUESTIONS CORRECT ***")
print()
print('Saved:', OUT_PATH)
